# Crib sheet: fitting tools

A quick reference for the fitting functions used in the Fanal exercise.  
Covers histogram fitting (`hfit`), extended likelihood fitting (`efit`),
profile likelihood scans, and confidence intervals.

## Setup

In [ ]:
%matplotlib inline

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os, sys
from pathlib import Path

_env = os.environ.get('FANAL_ROOT') or os.environ.get('USCFANALDIR')
if _env and Path(_env, 'data').is_dir():
    rootpath = str(Path(_env).resolve())
else:
    rootpath = str(next(p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'data').is_dir() and (p / 'ana').is_dir()))
if rootpath not in sys.path:
    sys.path.insert(0, rootpath)

import core.pltext  as pltext
import core.hfit    as hfit
import core.efit    as efit
import ana.fanal    as fn
import ana.pltfanal as pltfn
pltext.style()

print('Fanal root:', rootpath)

In [ ]:
# Load data for the examples
dirpath  = rootpath + '/data/'
filename = 'fanal_new_gamma.h5'

mcbi = pd.read_hdf(dirpath + filename, key = 'mc/bi214').fillna(0.)
mctl = pd.read_hdf(dirpath + filename, key = 'mc/tl208').fillna(0.)
mcbb = pd.read_hdf(dirpath + filename, key = 'mc/bb0nu').fillna(0.)
data_blind = pd.read_hdf(dirpath + filename, key = 'data/blind').fillna(0.)

sel_erange = (2.4, 2.7)  # MeV
sel_eroi   = (2.43, 2.48)

print('Bi: {:d}, Tl: {:d}, bb: {:d}, blind data: {:d} events'.format(
    len(mcbi), len(mctl), len(mcbb), len(data_blind)))

---
## 1. Histogram fitting with `hfit`

`hfit` fits analytic functions to binned data using least-squares.
The main function is `hfit.hfitres()` which fits **and** computes residuals.

### Available fit models

| Model | Function | Parameters | Description |
|-------|----------|------------|-------------|
| Gaussian | `hfit.fgaus` | $N, \mu, \sigma$ | Pure Gaussian peak |
| Line | `hfit.fline` | $a, b$ | Linear: $a x + b$ |
| Exponential | `hfit.fexp` | $N, \tau$ | $N e^{-\tau x}$ |
| Gaussian + line | `hfit.fgausline` | $N, \mu, \sigma, a, b$ | Peak on linear background |
| Gaussian + exp | `hfit.fgausexp` | $N, \mu, \sigma, N_\tau, \tau$ | Peak on exponential background |

Initial guesses are generated automatically for these built-in models.

### Example: fit a Gaussian to the $^{214}$Bi peak

The Bi energy spectrum in the analysis window has a peak on a background.
We fit a Gaussian + line model.

In [ ]:
# Select Bi events with 1 track in the energy range
sel = (mcbi.num_tracks == 1) & (mcbi.E >= sel_erange[0]) & (mcbi.E < sel_erange[1])
bins = 60

# Fit: returns (parameters, uncertainties, chi2, ndf)
par, epar, chi2, ndf = hfit.hfitres(mcbi.E[sel], bins,
                                     range = sel_erange,
                                     fun   = hfit.fgausline)

plt.xlabel('Energy (MeV)')
plt.title(r'$^{214}$Bi: Gaussian + line fit');

### Reading the fit results

For `hfit.fgausline`, the 5 parameters are:

| Index | Name | Meaning |
|-------|------|---------|
| 0 | $N$ | Normalization of the Gaussian |
| 1 | $\mu$ | Mean of the Gaussian |
| 2 | $\sigma$ | Standard deviation (width) |
| 3 | $a$ | Slope of the line |
| 4 | $b$ | Intercept of the line |

In [ ]:
# Extract parameters and uncertainties
mu    = par[1]
sigma = par[2]

print('Fit parameters:')
for i, (p, e) in enumerate(zip(par, epar)):
    print('  par[{:d}] = {:8.4f} +/- {:.4f}'.format(i, p, e))

print()
print('Peak position : {:.4f} +/- {:.4f} MeV'.format(mu, epar[1]))
print('Peak width    : {:.4f} +/- {:.4f} MeV'.format(sigma, epar[2]))
print('Resolution    : {:.2f} keV'.format(1000 * sigma))
print('Chi2 / ndf    : {:.1f} / {:d}'.format(chi2, ndf))

### Simpler variant: `hfit.hfit()`

If you only need the parameters (no residuals or $\chi^2$):

In [ ]:
# hfit.hfit returns just (pars, epars) — no chi2, no residual plot
par2, epar2 = hfit.hfit(mcbi.E[sel], bins, range = sel_erange, fun = hfit.fgausline)
print('sigma = {:.4f} +/- {:.4f} MeV'.format(par2[2], epar2[2]))

### Using `str_parameters()` for formatted output

Each model has a list of parameter names for display.

In [ ]:
# Pretty-print with LaTeX names
print(hfit.str_parameters(par, epar, parnames = hfit.ngausline))

---
## 2. Extended likelihood fitting

When you have multiple overlapping components (e.g. Bi + Tl backgrounds),
you can estimate how many events of each type are present using an
**extended maximum likelihood fit**.

The idea:
1. Build a PDF template for each component from MC simulation.
2. Combine them into an extended likelihood.
3. Minimize $-2 \log \mathcal{L}$ to find the best-fit event counts.

### Step 1: prepare the fit

`fn.prepare_fit_ell()` builds the likelihood from MC templates and returns
a **fit function** that you can call on any data.

In [ ]:
# Apply the blind selection to MC samples
mask_bi = fn.selection_blind(mcbi)
mask_tl = fn.selection_blind(mctl)
mc_blind = [mcbi[mask_bi], mctl[mask_tl]]

# Initial guess for the number of events
nguess = (1000., 1000.)

# Prepare the fit: returns a callable
fit = fn.prepare_fit_ell(mc_blind, nguess,
                         refnames  = ('E',),
                         refranges = (sel_erange,))

print('fit is a callable:', callable(fit))

### Step 2: run the fit on data

Call the fit function with a DataFrame.  
It returns `(result, values, ell, nevts_expected)`.

In [ ]:
# Fit the blind data
result, enes, ell, _ = fit(data_blind)

# result.x contains the best-fit event counts
n_Bi_blind = result.x[0]
n_Tl_blind = result.x[1]

print('Fitted number of Bi events: {:.1f}'.format(n_Bi_blind))
print('Fitted number of Tl events: {:.1f}'.format(n_Tl_blind))
print('Total                     : {:.1f} (observed: {:d})'.format(
    n_Bi_blind + n_Tl_blind, len(data_blind)))

### Step 3: plot the fit

`pltfn.plot_fit_ell()` shows the data histogram, the total fit, and
individual component contributions.

In [ ]:
sample_names_latex = [r'$^{214}$Bi', r'$^{208}$Tl']

pltfn.plot_fit_ell(enes, result.x, ell, parnames = sample_names_latex);

### What the fit function returns

| Return value | Type | Content |
|-------------|------|---------|
| `result` | `OptimizeResult` | `.x` = best-fit counts per component |
| `values` | `np.ndarray` | Data values used in the fit (e.g. energies) |
| `ell` | `ExtComPDF` | The likelihood object (needed for scans) |
| `nevts_exp` | `np.ndarray` | Expected counts after selection efficiency |

---
## 3. Profile likelihood scan and uncertainties

The uncertainty on a fitted parameter is obtained by scanning
$-2 \Delta \log \mathcal{L}$ around the minimum.  
The 68% CL interval corresponds to where $\Delta(-2 \log \mathcal{L}) = 1$.

### Run the scan

In [ ]:
# Scan all parameters: returns (list of scan points, list of tmu values)
nis, tmus = fn.tmu_scan(enes, result.x, ell, sizes = (3., 3.))

# nis[0] = scan points for parameter 0 (Bi)
# tmus[0] = corresponding -2 Delta log L values
print('Scan points for Bi: {:d} values from {:.0f} to {:.0f}'.format(
    len(nis[0]), nis[0][0], nis[0][-1]))

### Plot the scan

In [ ]:
pltfn.plot_tmu_scan(nis, tmus, titles = sample_names_latex);

### Extract confidence intervals

`efit.tmu_conf_int()` finds the parameter values where the scan crosses
a given confidence level.

In [ ]:
cl = 0.68  # 68% CL = 1 sigma

for i, name in enumerate(sample_names_latex):
    ci = efit.tmu_conf_int(nis[i], tmus[i], cl)
    n_est = result.x[i]
    print('{:s}: {:.1f}  ({:.1f}, +{:.1f})  at {:d}% CL'.format(
        name, n_est, ci[0] - n_est, ci[1] - n_est, int(100 * cl)))

### Manual scan with `efit.llike_scan()`

If you need finer control, you can scan one parameter yourself.

In [ ]:
# Scan parameter 0 (Bi) over a custom range
mu_scan = np.linspace(400, 620, 100)
tmu_vals = efit.llike_scan(enes, ell, result.x, mu_scan, ipos = 0)

plt.plot(mu_scan, tmu_vals)
plt.axhline(1, color='r', ls='--', label=r'$\Delta(-2\log L) = 1$')
plt.xlabel(r'$n^{\mathrm{Bi}}_b$')
plt.ylabel(r'$-2\Delta\log\mathcal{L}$')
plt.legend();

# Extract CI from this scan
ci_bi = efit.tmu_conf_int(mu_scan, tmu_vals, 0.68)
print('Bi 68% CI: ({:.1f}, {:.1f})'.format(*ci_bi))

---
## 4. Testing a signal hypothesis

In the signal notebooks, you fit 3 components (Bi + Tl + signal).
The test statistic $q_0$ tests the null hypothesis (no signal).

### Test statistic $q_0$

$$q_0 = -2 \log \frac{\mathcal{L}(n_{bb} = 0)}{\mathcal{L}(\hat{n}_{bb})}$$

The **discovery significance** is $Z_0 = \sqrt{q_0}$,
and the **p-value** is $p_0 = 1 - \Phi(Z_0)$.

In [ ]:
# Example: compute q0 for a fitted experiment
# (here we use a mock scenario — in the real exercise you use your fitted data)
#
# q0 = efit.tmu(values, ell, result.x, 0, ipos = 2)  # test n_bb = 0
# z0 = np.sqrt(q0)
# p0 = 1 - stats.norm.cdf(z0)
#
# print('q0 = {:.2f}'.format(q0))
# print('Z0 = {:.2f} sigma'.format(z0))
# print('p0 = {:.2e}'.format(p0))
print('(uncomment and adapt when you have a 3-component fit)')

### Half-life from the signal count

`fn.half_life()` converts a fitted number of signal events to a half-life.

In [ ]:
# fn.half_life(n_signal, exposure_kg_y, efficiency, abundance=0.9, W=135.9)
#
# Example:
# tau = fn.half_life(n_bb_est, exposure, eff_bb_E)
# print('Half-life: {:.2e} years'.format(tau))
print('(use with your fitted signal count, exposure, and efficiency)')

---
## Summary of the fitting workflow

```
1. Histogram fit (energy resolution)
   par, epar, chi2, ndf = hfit.hfitres(data, bins, range=..., fun=hfit.fgausline)
   sigma = par[2]

2. Extended likelihood fit (background estimation)
   fit = fn.prepare_fit_ell(mc_samples, nguess, refnames, refranges)
   result, values, ell, _ = fit(data)
   n_est = result.x          # fitted event counts

3. Profile likelihood scan (uncertainties)
   nis, tmus = fn.tmu_scan(values, n_est, ell, sizes=(3., 3.))
   ci = efit.tmu_conf_int(nis[i], tmus[i], cl=0.68)

4. Visualisation
   pltfn.plot_fit_ell(values, n_est, ell, parnames=...)
   pltfn.plot_tmu_scan(nis, tmus, titles=...)

5. Signal hypothesis
   q0 = efit.tmu(values, ell, result.x, 0, ipos=2)  # test n_bb = 0
   z0 = sqrt(q0)              # discovery significance
   tau = fn.half_life(n_bb, exposure, eff)  # half-life
```